[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DBertazioli/aml-labs/blob/main/lab1/2_competition_keras.ipynb)

# Lab 1, part 2: competition (Keras)

The data is the nested-shells data: 5 classes, 20 columns. Two formats, one live leaderboard.

- **speedrun**: any MLP, **at most 1000 optimizer steps** (a step is one update on one batch). **Batch size is fixed at 128.** The question is how much you can get out of a tiny training budget.
- **light**: any MLP with **at most 50,000 parameters** and **at most 40 epochs**. Batch size is free.

Rules:

- Any MLP: Dense layers, any activation, BatchNorm and Dropout are allowed; no convolutions or other model types. The model must be trained by you in this notebook. Any preprocessing computed from the training data is allowed.
- Honor system: the helpers below count steps, epochs, batch size and parameters from your real training, refuse a submission over budget, and record the numbers with every submission.
- Unlimited submissions (at most one every 20 seconds, counted separately for each leaderboard). The score you see is computed on **half of the test set**. The final ranking uses, for each student, the submission with the best visible score, scored on the **other half**, which you never see.
- Leaderboard: [speedrun and light](https://dbertazioli-aml-labs.hf.space/board?c=speedrun,light).

## 1. Setup

Downloads the class client and the data files. Run it once.

In [ ]:
import os, shutil, urllib.request
REPO = "DBertazioli/aml-labs"
RAW = f"https://raw.githubusercontent.com/{REPO}/main/"
LOCAL = os.environ.get("AML_LOCAL_REPO")  # only used when testing the notebook offline

def fetch(path, name):
    if LOCAL:
        shutil.copy(os.path.join(LOCAL, path), name)
    else:
        urllib.request.urlretrieve(RAW + path, name)

fetch("client/aml_client.py", "aml_client.py")
for name in ["shells_train.npz", "shells_test.npz"]:
    if not os.path.exists(name):
        fetch(f"lab1/competition/{name}", name)
import aml_client as aml

## 2. Register

Register with your campus email and a nickname. We do this together at the start of the competition.

- Your **campus email** (`@campus.unimib.it`) is used **only** to contact competition winners. It is **never shown** to anyone and it is **deleted after the course**.
- Your **nickname appears on the projector** in class, so **do not use your real name**. It has 2-20 characters: letters, digits, `_` or `-`.
- If the nickname is taken, pick another one.

The login is stored in a file. It survives "Restart session"; it is lost when Colab gives you a new runtime (disconnect, timeout, delete runtime). Then run this cell again with the same email and nickname (same spelling).

In [ ]:
EMAIL = "name.surname@campus.unimib.it"   # edit
NICKNAME = "..."                                # edit

if os.path.exists(aml.TOKEN_FILE):
    print("already registered")
else:
    if EMAIL == "name.surname@campus.unimib.it" or NICKNAME == "...":
        raise ValueError("Edit EMAIL and NICKNAME in this cell, then run it again.")
    aml.register(EMAIL, NICKNAME)

## 3. Data

Loads the training and test data. Look at the range of each column: some columns are in large units.

In [ ]:
import numpy as np

tr, te = np.load("shells_train.npz"), np.load("shells_test.npz")
X_train, y_train, X_test = tr["X_train"], tr["y_train"], te["X_test"]
K = int(y_train.max()) + 1
print("X_train", X_train.shape, "y_train", y_train.shape, "X_test", X_test.shape)
print("class counts:", np.bincount(y_train))
print("min per column:", np.round(X_train.min(0), 1))
print("max per column:", np.round(X_train.max(0), 1))
print("std per column:", np.round(X_train.std(0), 1))

## 4. The picture

The rule in 3-D: five nested shells, one inside the other, each a ball with a smaller ball on top. The class of a point is the shell it lies in: the innermost shell is class 0, the outermost class 4. Here the shells lie on their side and are cut in half, so you can see the inner shells.

In [ ]:
import plotly.graph_objects as go

colors = ["#0072b2", "#e69f00", "#009e73", "#cc79a7", "#d55e00"]

def rho(theta):   # radius of one shell at polar angle theta, for scale 1 (1 is a sphere)
    dc = 1.35 * np.cos(theta)
    disc = dc**2 - 1.35**2 + 0.6**2
    return np.where(disc >= 0, np.maximum(1, dc + np.sqrt(np.maximum(disc, 0))), 1.0)

def show(traces, eye):
    fig = go.Figure(traces)
    fig.update_layout(height=600, legend=dict(itemsizing="constant"),
                      scene=dict(xaxis_visible=False, yaxis_visible=False, zaxis_visible=False, aspectmode="data",
                                 camera=dict(eye=dict(x=eye[0], y=eye[1], z=eye[2]), up=dict(x=0, y=0, z=1))))
    fig.show()

# shell axis along x; phi in [pi, 2 pi] keeps the lower half (z <= 0), seen from above
th, ph = np.meshgrid(np.linspace(0, np.pi, 80), np.linspace(np.pi, 2 * np.pi, 60), indexing="ij")
shells = []
for k, c in enumerate(colors):
    r = (1.4 + 0.4 * k) * rho(th)          # outer surface of class k
    shells.append(go.Surface(x=r * np.cos(th), y=r * np.sin(th) * np.cos(ph), z=r * np.sin(th) * np.sin(ph),
                             surfacecolor=np.zeros(th.shape), colorscale=[[0, c], [1, c]], showscale=False,
                             opacity=1, lighting=dict(ambient=0.6, diffuse=0.7, specular=0.1, roughness=0.9),
                             name=f"class {k}", showlegend=True))
show(shells, (-0.8, 1.1, 1.1))

The same rule as points: this is what data from it look like in 3-D, one colour per class (a quarter is cut away so the inner shells show). The competition data uses the same rule in 20 dimensions, which cannot be drawn.

In [ ]:
rng = np.random.default_rng(0)
clouds = []
for k, c in enumerate(colors):
    n = 1870                                                   # about 1400 points are left after the cut
    s = rng.uniform(1 + 0.4 * k, 1.4 + 0.4 * k, n)             # scale inside the band of class k
    cos_t, phi = rng.uniform(-1, 1, n), rng.uniform(0, 2 * np.pi, n)
    sin_t = np.sqrt(1 - cos_t**2)
    r = s * rho(np.arccos(cos_t))
    x, y, z = r * sin_t * np.cos(phi), r * sin_t * np.sin(phi), r * cos_t
    keep = ~((x > 0) & (y > 0))                                # the cut quarter
    clouds.append(go.Scatter3d(x=x[keep], y=y[keep], z=z[keep], mode="markers", marker=dict(size=2, color=c),
                               name=f"class {k}"))
show(clouds, (1.5, 1.5, 1.0))

## 5. Budget helpers

Training goes through the provided function (`fit` in Keras, `train` in PyTorch). It counts the steps, epochs, samples and batch size of the real training and stores them on the model. Running a training cell again keeps counting on the same model: build a new model to start over. `submit` computes the number of parameters, checks the budget of the format, predicts the classes of `X_test` and sends the predictions. If you preprocess the inputs for training, pass the same function as `preprocess` so the test inputs are transformed the same way.

The same function plots the training loss after every epoch, so the plot updates while training runs (`LivePlot`, defined in the cell below). *Further reading: [Keras guide, writing your own callbacks](https://keras.io/guides/writing_your_own_callbacks/) (`KerasLive` is one)*

In [ ]:
import keras
import tensorflow as tf
from keras import layers

FORMATS = {"speedrun": "at most 1000 steps, batch size 128",
           "light": "at most 50,000 parameters, at most 40 epochs"}

import time
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

class LivePlot:
    """Keeps the per-epoch values and redraws the curves while training runs.
    update(logs) adds one epoch (redraw at most every `every` seconds); draw() shows the current state."""
    def __init__(self, every=0.5):
        self.h, self.every, self.t, self.title = {}, every, time.time(), ""   # h: (run, name) -> values per epoch
    def update(self, logs, run=""):                   # run: curve name when one plot shows several runs
        for k, v in logs.items():
            self.h.setdefault((run, k), []).append(float(v))
        n = len(self.h[(run, k)])
        self.title = f"{run + ': ' if run else ''}epoch {n} | " + " | ".join(f"{k} {float(v):.3f}" for k, v in logs.items())
        if time.time() - self.t > self.every:
            self.draw()
    def draw(self):
        if not self.h:
            return
        panels = [p for p in ("loss", "accuracy") if any(p in k for _, k in self.h)]
        fig, axes = plt.subplots(1, len(panels), figsize=(4.5 * len(panels), 3), squeeze=False)
        for ax, p in zip(axes[0], panels):
            for (run, k), v in self.h.items():
                if p in k:
                    ax.plot(range(1, len(v) + 1), v, label=run or k)
            ax.set_xlabel("epoch"); ax.set_title(p); ax.legend()
        fig.suptitle(self.title); fig.tight_layout()
        clear_output(wait=True); display(fig); plt.close(fig)
        self.t = time.time()

class KerasLive(keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.live = LivePlot()
    def on_epoch_end(self, epoch, logs=None):
        self.live.update(logs)
    def on_train_end(self, logs=None):
        self.live.draw()

class _Count(keras.callbacks.Callback):
    def __init__(self, max_steps):
        super().__init__()
        self.max_steps, self.epochs = max_steps, 0
        self.s0 = None
    def on_train_begin(self, logs=None):
        self.s0 = int(self.model.optimizer.iterations)
    def on_epoch_begin(self, epoch, logs=None):
        self.epochs += 1
    def on_train_batch_end(self, batch, logs=None):
        if self.max_steps and int(self.model.optimizer.iterations) - self.s0 >= self.max_steps:
            self.model.stop_training = True

def count_params(model):
    # all weights except the BatchNorm moving statistics (same count as in the PyTorch version)
    return int(sum(np.prod(v.shape) for v in model.weights if v.name not in ("moving_mean", "moving_variance")))

def fit(model, X, y, epochs=1, batch_size=128, max_steps=None):
    """Trains the model. Stops after `epochs` epochs or when the model has done `max_steps` steps in total.
    Steps, epochs, samples and batch size are added to the model (cumulative)."""
    done = getattr(model, "aml_steps", 0)       # steps of earlier fit calls, kept even if you recompile
    s0 = int(model.optimizer.iterations)
    if max_steps and done >= max_steps:
        print("step budget already used up, nothing to do")
        return
    cb = _Count(max_steps - done if max_steps else None)
    # a dataset that reshuffles every epoch (fixed seed, so a run can be repeated)
    data = tf.data.Dataset.from_tensor_slices((np.asarray(X, "float32"), y)).shuffle(len(X), seed=0, reshuffle_each_iteration=True).batch(batch_size)
    model.fit(data, epochs=epochs, shuffle=False, callbacks=[cb, KerasLive()], verbose=0)
    steps = int(model.optimizer.iterations) - s0
    model.aml_steps = done + steps
    model.aml_absmax = float(np.abs(np.asarray(X)).max())
    model.aml_epochs = getattr(model, "aml_epochs", 0) + cb.epochs
    model.aml_samples = getattr(model, "aml_samples", 0) + min(steps * batch_size, cb.epochs * len(X))
    model.aml_bs_min = min(getattr(model, "aml_bs_min", batch_size), batch_size)
    model.aml_bs_max = max(getattr(model, "aml_bs_max", batch_size), batch_size)
    print("total so far: steps", model.aml_steps, "| epochs", model.aml_epochs,
          "| batch size", model.aml_bs_max, "| params", count_params(model))

def submit(fmt, model, preprocess=lambda X: X):
    """fmt: "speedrun" or "light". Steps, epochs and batch size come from your training with fit()."""
    if fmt not in FORMATS:
        raise ValueError('fmt must be "speedrun" or "light"')
    if not hasattr(model, "aml_epochs"):
        raise ValueError("this model was not trained with fit(...): train it with the provided fit function")
    steps, epochs, params = max(getattr(model, "aml_steps", 0), int(model.optimizer.iterations)), model.aml_epochs, count_params(model)
    if fmt == "speedrun" and steps > 1000:
        raise ValueError(f"speedrun allows at most 1000 steps, this model has {steps} (steps add up over repeated training: build a new model)")
    if fmt == "speedrun" and (model.aml_bs_min, model.aml_bs_max) != (128, 128):
        raise ValueError(f"speedrun needs batch size 128, you used {model.aml_bs_min}-{model.aml_bs_max}")
    if fmt == "light" and params > 50_000:
        raise ValueError(f"light allows at most 50,000 parameters, your model has {params}")
    if fmt == "light" and (epochs > 40 or model.aml_samples > 40 * len(X_train)):
        raise ValueError(f"light allows at most 40 epochs, this model has {epochs} (epochs add up over repeated training: build a new model)")
    r = np.abs(preprocess(X_train)).max() / model.aml_absmax
    if r < 0.5 or r > 2:
        raise ValueError("The model was trained on preprocessed inputs: pass the same preprocess to submit(..., preprocess=...)")
    preds = model.predict(preprocess(X_test).astype("float32"), batch_size=1000, verbose=0).argmax(1)
    return aml.compete(fmt, preds, steps=steps, epochs=epochs, params=params,
                       batch_size=model.aml_bs_max, framework="keras")

## 6. Baseline

A small MLP (2 hidden layers of 64 ReLU units), plain SGD with learning rate 0.01, raw inputs. This is what you have to beat. The same model also fits the `light` budget (about 5.8k parameters, 7 epochs), so it is submitted to both.

In [ ]:
keras.utils.set_random_seed(0)
model = keras.Sequential([
    keras.Input((X_train.shape[1],)),
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(K),
])
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
fit(model, X_train, y_train, epochs=100, batch_size=128, max_steps=1000)   # speedrun: stops at 1000 steps
print("train accuracy:", model.evaluate(X_train, y_train, batch_size=1000, verbose=0)[1])
submit("speedrun", model)
submit("light", model)

## 7. Your turn

Write one attempt per format in the empty cell under its rules. A good start is the baseline of section 6: copy it, then change what the hints suggest. You can submit as often as you like, at most once every 20 s per leaderboard. Notebook 1, sections 8-10 (spiral arms, pitfalls, optimizer recap), cover the ideas below. Keep the names `fit`, `submit` and `preprocess` so the helpers still work.

### Speedrun

- **Budget**: at most 1000 optimizer steps (a step is one update on one batch) at **batch size 128**. That is about 6 passes over the training set.
- **Counting**: `fit` counts the steps of your real training, and `max_steps=1000` stops it there. Steps add up if you train the same model again: build a new model to start over.
- **Submit**: `submit("speedrun", model, preprocess=preprocess)` sends the predictions to the `speedrun` leaderboard.

Hints:

- **Input scaling**: look at the column ranges printed in section 3. Which direction was slow in the race, and why?
- **Learning rate and schedule**: the most sensitive knob when the budget is this small. Watch the live training loss. What should the step size be at the start and at the end?
- **Optimizer**: which ones coped with directions of very different curvature?

> **TODO.** In the cell below: build a model, train it with `fit(..., max_steps=1000)`, then submit it to `speedrun`.

### Light

- **Budget**: at most **50,000 parameters** and at most **40 epochs**. The batch size is free. Epochs add up if you train the same model again.
- **Layers**: Dense, any activation, BatchNorm, Dropout. No convolutions.
- **Submit**: `submit("light", model, preprocess=preprocess)` sends the predictions to the `light` leaderboard. It counts the parameters with `count_params` (all weights and biases, BatchNorm scale and shift included) and refuses a model over budget.

Hints:

- **Width and depth**: a Dense layer with n inputs and m units has (n + 1) * m parameters. Wider or deeper, which is better for 50,000 parameters?
- **BatchNorm**: how does it change the learning rate you can use?
- **Longer training with a schedule**: 40 epochs is about 6 times the speedrun budget. What should the step size be at the start and at the end?
- **Feature engineering**: any transformation computed from the training data is allowed, if you pass the same function as `preprocess`. What would make the classes easier to separate?

> **TODO.** In the cell below: build a model with at most 50,000 parameters, train it with `fit` for at most 40 epochs, then submit it to `light`.